In [1]:
# ============================================
# IMPORT LIBRARIES
# ============================================

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import cifar10

2026-05-10 14:34:50.946433: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778423691.204759      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778423691.276089      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778423691.943067      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778423691.943122      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778423691.943127      57 computation_placer.cc:177] computation placer alr

In [2]:
# ============================================
# LOAD CIFAR-10 DATASET (TESTING)
# ============================================

(x_train, y_train), (x_test, y_test) = cifar10.load_data()

print("CIFAR Test Shape:", x_test.shape)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
CIFAR Test Shape: (10000, 32, 32, 3)


In [3]:
# ============================================
# NORMALIZE CIFAR TEST DATA
# ============================================

x_test = x_test.astype("float32") / 255.0

y_test = y_test.flatten()

In [4]:
# ============================================
# LOAD CINIC-10 TRAIN DATASET
# ============================================

cinic_train = tf.keras.utils.image_dataset_from_directory(
    '/kaggle/input/datasets/mengcius/cinic10/train',
    image_size=(32,32),
    batch_size=64,
    shuffle=True
)

Found 90000 files belonging to 10 classes.


2026-05-10 14:38:23.727687: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [5]:
# ============================================
# NORMALIZE CINIC TRAIN DATA
# ============================================

cinic_train = cinic_train.map(
    lambda x, y: (
        tf.cast(x, tf.float32) / 255.0,
        tf.cast(y, tf.int32)
    )
)

In [6]:
# ============================================
# DATA AUGMENTATION
# ============================================

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

In [7]:
# ============================================
# XCEPTION BLOCK
# ============================================

def xception_block(x, filters):

    residual = layers.Conv2D(
        filters,
        (1,1),
        strides=2,
        padding='same'
    )(x)

    x = layers.SeparableConv2D(
        filters,
        (3,3),
        padding='same',
        activation='relu'
    )(x)

    x = layers.BatchNormalization()(x)

    x = layers.SeparableConv2D(
        filters,
        (3,3),
        padding='same',
        activation='relu'
    )(x)

    x = layers.BatchNormalization()(x)

    x = layers.MaxPooling2D(
        (3,3),
        strides=2,
        padding='same'
    )(x)

    x = layers.Add()([x, residual])

    return x

In [8]:
# ============================================
# BUILD XCEPTION MODEL
# ============================================

def build_xception(input_shape=(32,32,3), num_classes=10):

    inputs = layers.Input(shape=input_shape)

    x = data_augmentation(inputs)

    x = layers.Conv2D(
        32,
        (3,3),
        padding='same',
        activation='relu'
    )(x)

    x = xception_block(x, 64)

    x = xception_block(x, 128)

    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(128, activation='relu')(x)

    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(
        num_classes,
        activation='softmax'
    )(x)

    model = models.Model(inputs, outputs)

    return model

In [9]:
# ============================================
# CREATE MODEL
# ============================================

model = build_xception()

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 32, 32, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential          │ (None, 32, 32, 3) │          0 │ input_layer[0][0] │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 32, 32,    │        896 │ sequential[0][0]  │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ separable_conv2d    │ (None, 32, 32,    │      2,400 │ conv2d[0][0]      │
│ (SeparableConv2D)   │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 32, 32,    │        256 │ separable_conv2d… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ separable_conv2d_1  │ (None, 32, 32,    │      4,736 │ batch_normalizat… │
│ (SeparableConv2D)   │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        256 │ separable_conv2d… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 16, 16,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 16, 16,    │      2,112 │ conv2d[0][0]      │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 16, 16,    │          0 │ max_pooling2d[0]… │
│                     │ 64)               │            │ conv2d_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ separable_conv2d_2  │ (None, 16, 16,    │      8,896 │ add[0][0]         │
│ (SeparableConv2D)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 16, 16,    │        512 │ separable_conv2d… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ separable_conv2d_3  │ (None, 16, 16,    │     17,664 │ batch_normalizat… │
│ (SeparableConv2D)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 16, 16,    │        512 │ separable_conv2d… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 8, 8, 128) │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 8, 8, 128) │      8,320 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 8, 8, 128) │          0 │ max_pooling2d_1[… │
│                     │                   │            │ conv2d_2[0][0]  

 Total params: 64,362 (251.41 KB)

 Trainable params: 63,594 (248.41 KB)

 Non-trainable params: 768 (3.00 KB)

In [10]:
# ============================================
# COMPILE MODEL
# ============================================

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [11]:
# ============================================
# TRAIN ON CINIC-10
# ============================================

history = model.fit(
    cinic_train,
    epochs=40,
    validation_data=(x_test, y_test)
)

Epoch 1/40
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 389s 272ms/step - accuracy: 0.3141 - loss: 1.8496 - val_accuracy: 0.4726 - val_loss: 1.4167
Epoch 2/40
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 364s 258ms/step - accuracy: 0.4554 - loss: 1.4803 - val_accuracy: 0.5175 - val_loss: 1.3451
Epoch 3/40
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 365s 260ms/step - accuracy: 0.5013 - loss: 1.3659 - val_accuracy: 0.5797 - val_loss: 1.1688
Epoch 4/40
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 366s 260ms/step - accuracy: 0.5289 - loss: 1.2871 - val_accuracy: 0.6095 - val_loss: 1.1066
Epoch 5/40
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 363s 258ms/step - accuracy: 0.5512 - loss: 1.2386 - val_accuracy: 0.6658 - val_loss: 0.9757
Epoch 6/40
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 367s 261ms/step - accuracy: 0.5702 - loss: 1.1961 - val_accuracy: 0.6589 - val_loss: 1.0086
Epoch 7/40
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 376s 257ms/step - accuracy: 0.5806 - loss: 1.1661 - val_accuracy: 0.6656 - val_loss: 0.9697
Epoch 8/40
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 358s 254ms/step - ac

In [12]:
# ============================================
# TEST ON CIFAR-10
# ============================================

test_loss, test_acc = model.evaluate(x_test, y_test)

print("CINIC → CIFAR Xception Accuracy:", test_acc)

313/313 ━━━━━━━━━━━━━━━━━━━━ 8s 25ms/step - accuracy: 0.7628 - loss: 0.7041
CINIC → CIFAR Xception Accuracy: 0.7627999782562256


In [13]:
# ============================================
# SAVE MODEL
# ============================================

model.save('/kaggle/working/cinic_to_cifar_xception.keras')